# Etape 3 - Modelisation fraude (Machine Learning)

Notebook de depart pour predire `fraud_reported` a partir de `insurance_claims.csv`.


In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.model_selection import RandomizedSearchCV


## 2) Chargement des donnees


In [21]:
DATA_PATH = '../process/insurance_claims_cleaned.csv'
df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
df.head()


Shape: (1000, 36)


,months_as_customer,age,policy_bind_date,policy_state,policy_csl,policy_deductable,policy_annual_premium,umbrella_limit,insured_sex,insured_education_level,...,witnesses,police_report_available,total_claim_amount,injury_claim,property_claim,vehicle_claim,auto_make,auto_model,auto_year,fraud_reported
0,328,48,2014-10-17,OH,250/500,1000,1406.91,0,MALE,MD,...,2,YES,71610,6510,13020,52080,SAAB,92X,2004,Y
1,228,42,2006-06-27,IN,250/500,2000,1197.22,5000000,MALE,MD,...,0,NO,5070,780,780,3510,MERCEDES,E400,2007,Y
2,134,29,2000-09-06,OH,100/300,2000,1413.14,5000000,FEMALE,PHD,...,3,NO,34650,7700,3850,23100,DODGE,RAM,2007,N
3,256,41,1990-05-25,IL,250/500,2000,1415.74,6000000,FEMALE,PHD,...,2,NO,63400,6340,6340,50720,CHEVROLET,TAHOE,2014,Y
4,228,44,2014-06-06,IL,500/1000,1000,1583.91,6000000,MALE,ASSOCIATE,...,1,NO,6500,1300,650,4550,ACCURA,RSX,2009,N


## 3) Nettoyage minimum pour modelisation


In [22]:
# Preprocessing & Feature Engineering Expert
if '_c39' in df.columns:
    df = df.drop(columns=['_c39'])
df = df.replace('?', np.nan)

# Conversion dates
for c in ['policy_bind_date', 'incident_date']:
    df[c] = pd.to_datetime(df[c], errors='coerce')

# 1. Relation 1 : Early Incident (< 30 jours)
df['claim_delay_days'] = (df['incident_date'] - df['policy_bind_date']).dt.days
df['rel1_early_incident'] = (df['claim_delay_days'] < 30).astype(int)

# Seuils statistiques pour les calculs suivants
high_amount = df['total_claim_amount'].quantile(0.75)
low_premium = df['policy_annual_premium'].quantile(0.25)

# 2. Relation 2 : Gravité faible + montant élevé
low_severity = df['incident_severity'].isin(['Minor Damage', 'Trivial Damage'])
df['rel2_severity_amount_mismatch'] = (low_severity & (df['total_claim_amount'] > high_amount)).astype(int)

# 3. Relation 3 : Proximité franchise (Franchise < Montant < Franchise + 1000)
df['rel3_just_above_deductible'] = ((df['total_claim_amount'] > df['policy_deductable']) & 
                                    (df['total_claim_amount'] < df['policy_deductable'] + 1000)).astype(int)

# 4. Relation 4 : Pas de rapport de police + montant élevé
df['rel4_no_police_high_amount'] = ((df['police_report_available'] == 'NO') & 
                                    (df['total_claim_amount'] > high_amount)).astype(int)

# 5. Relation 5 : Petite prime + énorme réclamation
df['rel5_low_premium_high_claim'] = ((df['policy_annual_premium'] < low_premium) & 
                                     (df['total_claim_amount'] > high_amount)).astype(int)

# 6. Relation 6 : Nuit + Single Vehicle
is_night = df['incident_hour_of_the_day'].between(23, 23) | df['incident_hour_of_the_day'].between(0, 5)
df['rel6_night_single_vehicle'] = (is_night & (df['incident_type'] == 'Single Vehicle Collision')).astype(int)

# Features temporelles classiques de base
df['incident_month'] = df['incident_date'].dt.month
df['policy_year'] = df['policy_bind_date'].dt.year

df.head()


,months_as_customer,age,policy_state,policy_csl,policy_deductable,policy_annual_premium,umbrella_limit,insured_sex,insured_education_level,insured_occupation,...,police_report_available,total_claim_amount,injury_claim,property_claim,vehicle_claim,auto_make,auto_model,auto_year,fraud_reported,claim_delay_days
0,328,48,OH,250/500,1000,1406.91,0,MALE,MD,CRAFT-REPAIR,...,YES,71610,6510,13020,52080,SAAB,92X,2004,Y,100
1,228,42,IN,250/500,2000,1197.22,5000000,MALE,MD,MACHINE-OP-INSPCT,...,NO,5070,780,780,3510,MERCEDES,E400,2007,Y,3130
2,134,29,OH,100/300,2000,1413.14,5000000,FEMALE,PHD,SALES,...,NO,34650,7700,3850,23100,DODGE,RAM,2007,N,5282
3,256,41,IL,250/500,2000,1415.74,6000000,FEMALE,PHD,ARMED-FORCES,...,NO,63400,6340,6340,50720,CHEVROLET,TAHOE,2014,Y,8996
4,228,44,IL,500/1000,1000,1583.91,6000000,MALE,ASSOCIATE,SALES,...,NO,6500,1300,650,4550,ACCURA,RSX,2009,N,256


## 4) Definition X / y + split


In [23]:
target = 'fraud_reported'
y = df[target].map({'N': 0, 'Y': 1})

# On garde vos 10 colonnes + les flags experts + les 3 colonnes boosters
selected_features = [
    # Votre sélection initiale
    'total_claim_amount', 'policy_deductable', 'policy_annual_premium', 
    'incident_severity', 'police_report_available', 'incident_type', 
    'witnesses', 'bodily_injuries', 'claim_delay_days', 
    'incident_month', 'policy_year', 
    # Les relation expertes (Rel 1 à 6)
    'rel1_early_incident', 'rel2_severity_amount_mismatch', 
    'rel3_just_above_deductible', 'rel4_no_police_high_amount', 
    'rel5_low_premium_high_claim', 'rel6_night_single_vehicle',
    # Les colonnes boosters (forte puissance prédictive)
    'age', 'months_as_customer', 'insured_hobbies'
]

X = df[selected_features]

print('Fraud ratio:', y.mean().round(3))
print('Nombre total de features:', len(selected_features))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)

print('Train:', X_train.shape, 'Test:', X_test.shape)


Fraud ratio: 0.247
Train: (600, 34) Test: (400, 34)


## 5) Pipeline preprocessing


In [1]:
num_cols = X.select_dtypes(include=['number']).columns.tolist()
cat_cols = X.select_dtypes(exclude=['number']).columns.tolist()

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocess = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])

print('Numeriques:', len(num_cols), 'Categoriels:', len(cat_cols))


NameError: name 'X' is not defined

## 6) Baseline 1 - Logistic Regression


In [29]:
logreg_model = Pipeline([
    ('prep', preprocess),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

logreg_model.fit(X_train, y_train)
pred_lr = logreg_model.predict(X_test)
proba_lr = logreg_model.predict_proba(X_test)[:, 1]

print('=== Logistic Regression ===')
print(classification_report(y_test, pred_lr, digits=3))
print('Confusion matrix:\n', confusion_matrix(y_test, pred_lr))
print('ROC-AUC:', round(roc_auc_score(y_test, proba_lr), 3))


=== Logistic Regression ===
              precision    recall  f1-score   support

           0      0.784     0.568     0.659       301
           1      0.286     0.525     0.370        99

    accuracy                          0.557       400
   macro avg      0.535     0.547     0.515       400
weighted avg      0.661     0.557     0.587       400

Confusion matrix:
 [[171 130]
 [ 47  52]]
ROC-AUC: 0.582


## 7) Baseline 2 - Random Forest


In [26]:
rf_model = Pipeline([
    ('prep', preprocess),
    ('clf', RandomForestClassifier(
        n_estimators=400,
        random_state=42,
        class_weight='balanced_subsample',
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)
pred_rf = rf_model.predict(X_test)
proba_rf = rf_model.predict_proba(X_test)[:, 1]

print('=== Random Forest ===')
print(classification_report(y_test, pred_rf, digits=3))
print('Confusion matrix:\n', confusion_matrix(y_test, pred_rf))
print('ROC-AUC:', round(roc_auc_score(y_test, proba_rf), 3))


=== Random Forest ===
              precision    recall  f1-score   support

           0      0.798     0.930     0.859       301
           1      0.571     0.283     0.378        99

    accuracy                          0.770       400
   macro avg      0.685     0.607     0.619       400
weighted avg      0.742     0.770     0.740       400

Confusion matrix:
 [[280  21]
 [ 71  28]]
ROC-AUC: 0.819


In [30]:
from xgboost import XGBClassifier

# Pipeline XGBoost
xgb_model = Pipeline([
    ('prep', preprocess),
    ('clf', XGBClassifier(
        n_estimators=1000,
        max_depth=5,
        learning_rate=0.1,
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),  # pour déséquilibre
        use_label_encoder=False,
        eval_metric='logloss',
         # Bonus importants:
        subsample=0.8,      # Échantillonnage des données (0.5-1.0)
        colsample_bytree=0.8,  # Échantillonnage des features
        min_child_weight=1,    # Régularisation
        random_state=42
    ))
])

xgb_model.fit(X_train, y_train)
pred_xgb = xgb_model.predict(X_test)
proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

print('=== XGBoost ===')
print(classification_report(y_test, pred_xgb, digits=3))
print('Confusion matrix:\n', confusion_matrix(y_test, pred_xgb))
print('ROC-AUC:', round(roc_auc_score(y_test, proba_xgb), 3))

=== XGBoost ===
              precision    recall  f1-score   support

           0      0.892     0.880     0.886       301
           1      0.650     0.677     0.663        99

    accuracy                          0.830       400
   macro avg      0.771     0.779     0.775       400
weighted avg      0.832     0.830     0.831       400

Confusion matrix:
 [[265  36]
 [ 32  67]]
ROC-AUC: 0.837


## 7bis) Modèle Arbre - Decision Tree
Nous ajoutons un arbre de décision simple pour illustrer la famille des modèles à base d'arbre.

In [31]:
from sklearn.tree import DecisionTreeClassifier

# Pipeline Decision Tree
dt_model = Pipeline([
    ('prep', preprocess),
    ('clf', DecisionTreeClassifier(
        max_depth=6,
        class_weight='balanced',
        random_state=42
))
])

dt_model.fit(X_train, y_train)
pred_dt = dt_model.predict(X_test)
proba_dt = dt_model.predict_proba(X_test)[:, 1]

print('=== Decision Tree ===')
print(classification_report(y_test, pred_dt, digits=3))
print('Confusion matrix:\n', confusion_matrix(y_test, pred_dt))
print('ROC-AUC:', round(roc_auc_score(y_test, proba_dt), 3))

=== Decision Tree ===
              precision    recall  f1-score   support

           0      0.909     0.831     0.868       301
           1      0.592     0.747     0.661        99

    accuracy                          0.810       400
   macro avg      0.751     0.789     0.764       400
weighted avg      0.831     0.810     0.817       400

Confusion matrix:
 [[250  51]
 [ 25  74]]
ROC-AUC: 0.776


## 8) Modèle Avancé - CatBoost
CatBoost est particulièrement performant avec les données catégorielles.

In [ ]:
cat_model = Pipeline([
    ('prep', preprocess),
    ('clf', CatBoostClassifier(iterations=500, learning_rate=0.1, depth=6, verbose=0, random_state=42, auto_class_weights='Balanced'))
])

cat_model.fit(X_train, y_train)
pred_cat = cat_model.predict(X_test)
proba_cat = cat_model.predict_proba(X_test)[:, 1]

print('=== CatBoost ===')
print(classification_report(y_test, pred_cat, digits=3))
print('ROC-AUC:', round(roc_auc_score(y_test, proba_cat), 3))

## 9) Modèle Avancé - LightGBM
LightGBM est connu pour sa rapidité et son efficacité sur les grands datasets.

In [ ]:
lgb_model = Pipeline([
    ('prep', preprocess),
    ('clf', LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=31, random_state=42, verbose=-1))
])

lgb_model.fit(X_train, y_train)
pred_lgb = lgb_model.predict(X_test)
proba_lgb = lgb_model.predict_proba(X_test)[:, 1]

print('=== LightGBM ===')
print(classification_report(y_test, pred_lgb, digits=3))
print('ROC-AUC:', round(roc_auc_score(y_test, proba_lgb), 3))

## 10) Comparaison Finale
Comparons les modèles sur le ROC-AUC.

In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics import precision_recall_curve, f1_score, roc_auc_score

def evaluate_recall_target(name, y_true, y_proba, target_recall=0.85):
    try:
        # Calcul des courbes
        precisions, recalls, thresholds = precision_recall_curve(y_true, y_proba)
        
        # Trouver le seuil où le recall est au moins égal au target
        indices = np.where(recalls >= target_recall)[0]
        if len(indices) == 0:
            best_idx = 0
        else:
            best_idx = indices[-1]
        
        # Securite pour l'index
        idx_th = min(best_idx, len(thresholds)-1)
        best_th = thresholds[idx_th]
        
        # Métriques au seuil cible
        rec_final = recalls[best_idx]
        prec_final = precisions[best_idx]
        f1_final = 2 * (prec_final * rec_final) / (prec_final + rec_final + 1e-10)
        
        return {
            'Model': name,
            'ROC-AUC': round(roc_auc_score(y_true, y_proba), 3),
            'Targeted_Threshold': round(best_th, 3),
            'Recall_Obtained': round(rec_final, 3),
            'Precision': round(prec_final, 3),
            'F1_Score': round(f1_final, 3)
        }
    except Exception as e:
        return {'Model': name, 'Error': str(e)}

# Comparaison avec cible de 85% de Rappel
try:
    target = 0.85
    # On verifie que les variables existent avant de comparer
    model_probas = {
        'LogisticReg': globals().get('proba_lr'),
        'RandomForest': globals().get('proba_rf'),
        'DecisionTree': globals().get('proba_dt'),
        'XGBoost_Tuned': globals().get('proba_xgb_tuned'),
        'CatBoost_Tuned': globals().get('proba_cat_tuned')
    }
    
    recall_results = []
    for name, proba in model_probas.items():
        if proba is not None:
            recall_results.append(evaluate_recall_target(name, y_test, proba, target_recall=target))
        else:
            print(f"Attention: {name} n'est pas encore entraîne (proba manquante).")

    if recall_results:
        df_recall = pd.DataFrame(recall_results).sort_values('Precision', ascending=False)
        print(f'\n=== Comparaison pour un objectif de {target*100}% de Rappel ===')
        print(df_recall.to_markdown(index=False))
    else:
        print("Erreur: Aucun modèle n'est prêt pour la comparaison. Veuillez lancer les entraînements.")
except Exception as e:
    print(f"Erreur critique : {e}")


Erreur : name 'proba_lr' is not defined
Veuillez vous assurer d'avoir exécuté toutes les cellules précédentes (entraînement des modèles).


## 11) Optimisation Avancée (SMOTE + Tuning)
Nous allons utiliser SMOTE pour équilibrer les classes et RandomizedSearchCV pour trouver les meilleurs paramètres.

In [ ]:
# 11.1) XGBoost Tuned avec SMOTE
xgb_tuned_pipe = ImbPipeline([
    ('prep', preprocess),
    ('smote', SMOTE(random_state=42)),
    ('clf', XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'))
])

param_grid_xgb = {
    'clf__n_estimators': [100, 500, 1000],
    'clf__max_depth': [3, 5, 7, 9],
    'clf__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'clf__subsample': [0.6, 0.8, 1.0],
    'clf__colsample_bytree': [0.6, 0.8, 1.0]
}

xgb_search = RandomizedSearchCV(
    xgb_tuned_pipe, param_distributions=param_grid_xgb, 
    n_iter=10, cv=3, scoring='roc_auc', n_jobs=-1, random_state=42, verbose=1
)

xgb_search.fit(X_train, y_train)
best_xgb = xgb_search.best_estimator_

print('Meilleurs paramètres XGB:', xgb_search.best_params_)
pred_xgb_tuned = best_xgb.predict(X_test)
proba_xgb_tuned = best_xgb.predict_proba(X_test)[:, 1]

print('\n=== XGBoost Tuned + SMOTE ===')
print(classification_report(y_test, pred_xgb_tuned, digits=3))
print('ROC-AUC:', round(roc_auc_score(y_test, proba_xgb_tuned), 3))

In [ ]:
# 11.1b) CatBoost Tuned avec SMOTE
cat_tuned_pipe = ImbPipeline([ 
    ('prep', preprocess),
    ('smote', SMOTE(random_state=42)),
    ('clf', CatBoostClassifier(verbose=0, random_state=42, auto_class_weights='Balanced'))
])

param_grid_cat = {
    'clf__iterations': [100, 500, 1000],
    'clf__depth': [4, 6, 8],
    'clf__learning_rate': [0.01, 0.05, 0.1],
    'clf__l2_leaf_reg': [1, 3, 5, 7]
}

cat_search = RandomizedSearchCV(
    cat_tuned_pipe, param_distributions=param_grid_cat, 
    n_iter=5, cv=3, scoring='roc_auc', n_jobs=-1, random_state=42, verbose=1
)

cat_search.fit(X_train, y_train)
best_cat = cat_search.best_estimator_

print('Meilleurs paramètres CatBoost:', cat_search.best_params_)
pred_cat_tuned = best_cat.predict(X_test)
proba_cat_tuned = best_cat.predict_proba(X_test)[:, 1]

print('\n=== CatBoost Tuned + SMOTE ===')
print(classification_report(y_test, pred_cat_tuned, digits=3))
print('ROC-AUC:', round(roc_auc_score(y_test, proba_cat_tuned), 3))

In [ ]:
# 11.2) Optimisation du Seuil (Threshold Tuning)
from sklearn.metrics import precision_recall_curve

precisions, recalls, thresholds = precision_recall_curve(y_test, proba_xgb_tuned)

# Trouver le seuil qui maximise le F1-score
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f'Meilleur seuil (F1-score): {best_threshold:.3f}')

# Nouvelle prédiction avec le seuil optimisé
pred_custom = (proba_xgb_tuned >= best_threshold).astype(int)

print('\n=== XGBoost Tuned (Seuil Optimisé) ===')
print(classification_report(y_test, pred_custom, digits=3))

## 8) Prochaine etape
- Ajuster le seuil de decision pour augmenter le recall fraude.
- Ajouter SHAP ou importances pour expliquer les predictions.
- Sauvegarder le meilleur pipeline (`joblib`).
